# 02 — The Agent Card: How Agents Describe Themselves

## Why this notebook exists

In **notebook 01** we built two services that talked to each other through hand-rolled REST. We watched that fall over the moment one side changed its contract — because there was no machine-readable description of either service. Consumers had no way to ask: *"what skills do you have, what inputs do you take, and how do I authenticate?"*

A2A solves this with a single, simple primitive: the **Agent Card**. It's a JSON document served at a well-known URL (`/.well-known/agent.json`) that describes everything a client needs to know to start talking to an agent.

This notebook builds an Agent Card for the researcher from notebook 01, fetches it from a client, and walks through what the card actually says.

## What you'll learn

- The shape of an A2A **Agent Card**: name, description, URL, capabilities, authentication, skills.
- How to serve `/.well-known/agent.json` from a FastAPI app.
- How to discover an agent's capabilities from the client side using `httpx`.
- How to parse a card into a typed `pydantic` model and iterate over its declared skills.
- Why the Agent Card is **not** the same as OpenAPI, and what each is good for.

## 1. Setup

Same pattern as notebook 01 — we run a FastAPI app on a background thread so we can both serve and call it from this notebook. Each notebook in this series is self-contained, so the helper is re-defined here rather than imported.

In [ ]:
import json
import threading
import time

import httpx
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    """Start `app` on localhost:`port` in a background daemon thread."""
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")

    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")

## 2. What is an Agent Card?

An Agent Card is a JSON document an agent **publishes about itself**. The A2A spec mandates it lives at the path `/.well-known/agent.json` on the agent's base URL — the same `/.well-known/` convention used by OAuth, OpenID Connect, Web App Manifests, and other web standards.

A minimal card answers four questions:

| Question | Card field |
|---|---|
| *Who are you?* | `name`, `description`, `version` |
| *Where do I send requests?* | `url` |
| *What can you do?* | `capabilities`, `skills[]`, `defaultInputModes`, `defaultOutputModes` |
| *How do I authenticate?* | `authentication.schemes[]` |

The card is **machine-readable** — a client fetches it once at the start of an interaction and can then make sensible decisions: "this agent supports streaming, so I'll use `message/stream` instead of `message/send`," or "this agent requires a bearer token, so I'll attach one."

The card is **also human-readable** — `description` and `skills[].description` are free-form prose meant to help humans (and LLMs reading the card on their behalf) understand what the agent is for.

## 3. Serve the Researcher's Agent Card

Now we build the server side. We'll:

1. Define `pydantic` models matching the A2A card schema (the shapes are stable enough to type).
2. Build a FastAPI app with one endpoint: `GET /.well-known/agent.json`.
3. Start it on `127.0.0.1:8010`.

The card declares one skill — `research_topic` — corresponding to the researcher's only ability. It also declares **`authentication.schemes: ["none"]`** because we haven't introduced auth yet (notebook 07 will).

In [ ]:
class AgentCapabilities(BaseModel):
    streaming: bool = False
    pushNotifications: bool = False
    stateTransitionHistory: bool = False


class AgentAuthentication(BaseModel):
    schemes: list[str] = Field(default_factory=lambda: ["none"])


class AgentSkill(BaseModel):
    id: str
    name: str
    description: str
    tags: list[str] = Field(default_factory=list)
    examples: list[str] = Field(default_factory=list)


class AgentCard(BaseModel):
    name: str
    description: str
    url: str
    version: str
    capabilities: AgentCapabilities = Field(default_factory=AgentCapabilities)
    authentication: AgentAuthentication = Field(default_factory=AgentAuthentication)
    defaultInputModes: list[str] = Field(default_factory=lambda: ["text"])
    defaultOutputModes: list[str] = Field(default_factory=lambda: ["text"])
    skills: list[AgentSkill] = Field(default_factory=list)


researcher_app = FastAPI()


RESEARCHER_CARD = AgentCard(
    name="Researcher",
    description="Returns canned facts on a small set of well-known topics.",
    url="http://127.0.0.1:8010",
    version="0.1.0",
    skills=[
        AgentSkill(
            id="research_topic",
            name="Research a topic",
            description="Given a topic name, return a list of facts about it.",
            tags=["research", "facts"],
            examples=["octopuses", "rome"],
        ),
    ],
)


@researcher_app.get("/.well-known/agent.json", response_model=AgentCard)
def agent_card() -> AgentCard:
    return RESEARCHER_CARD


researcher_server = run_server_in_thread(researcher_app, port=8010)
print("Researcher running on http://127.0.0.1:8010")

In [ ]:
resp = httpx.get("http://127.0.0.1:8010/.well-known/agent.json")
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## 4. Discover the Agent from a Client

A client that wants to talk to an agent does one thing first: **fetch the Agent Card**. With nothing more than the agent's base URL, it can learn the agent's identity, capabilities, and skills.

We'll parse the card back into the same `AgentCard` pydantic model we defined on the server side. In a real cross-team setup the client wouldn't have access to the server's classes — it would have its own definitions of the same A2A schema, or use a shared SDK. The shapes are governed by the A2A spec, not by either side's code.

In [ ]:
def discover_agent(base_url: str) -> AgentCard:
    """Fetch and parse an agent's card from its base URL."""
    resp = httpx.get(f"{base_url.rstrip('/')}/.well-known/agent.json")
    resp.raise_for_status()
    return AgentCard.model_validate(resp.json())


card = discover_agent("http://127.0.0.1:8010")
print(f"Found agent: {card.name} (v{card.version})")
print(f"  Description: {card.description}")
print(f"  Endpoint:    {card.url}")
print(f"  Auth:        {card.authentication.schemes}")
print(f"  Streaming?   {card.capabilities.streaming}")
print(f"  # skills:    {len(card.skills)}")